In [1]:
#load data/interpo.xml using xml parsing library
import xml.etree.ElementTree as ET
tree = ET.parse('data/interpro.xml')
root = tree.getroot()

In [2]:
#print number of children of root
interpro_accession_l = []
interpro_member_db_l = []
interpro_type_l = []
prot_count_l = []

count = 0
for entry in root.findall('interpro'):
    memberlist = entry.find('member_list')
    db_entry = [db.get('db') for db in memberlist.findall('db_xref')]
    entry_type = entry.get('type')
    interpro_type_l.append(entry_type)
    interpro_accession_l.append(entry.get('id'))
    interpro_member_db_l.append(db_entry)
    num_prot = int(entry.get('protein_count'))
    prot_count_l.append(num_prot)
    count += 1

from itertools import chain
all_db = set(chain.from_iterable(interpro_member_db_l))
db_cols = {'interpro_accession': interpro_accession_l, 'type': interpro_type_l, 'protein_count': prot_count_l}
for db in all_db:
    db_cols[db] = [True if db in db_entry else False for db_entry in interpro_member_db_l]
import pandas as pd
interpro_df = pd.DataFrame(db_cols)


In [3]:
interpro_df = interpro_df[interpro_df['protein_count'] > 1e4]
go_annot = pd.read_csv('data/interpro2go', sep=';', header=None, names=['interpro_accession', 'go_id'], comment='!')
go_annot['go_id'] = go_annot['go_id'].str.strip()
go_set = set(go_annot['go_id'])
#using regex, extract id from string of form "InterPro:{ID}..."
go_annot['interpro_accession'] = go_annot['interpro_accession'].str.extract(r'InterPro:(\S+)')
go_annot = go_annot.groupby('interpro_accession').agg({'go_id': list}).reset_index()
interpro_df['go_term'] = interpro_df['interpro_accession'].map(dict(zip(go_annot['interpro_accession'], go_annot['go_id'])))
interpro_df.dropna(subset=['go_term'], inplace=True)
interpro_df = interpro_df[~interpro_df['go_term'].apply(lambda x: len(x) == 1 and 'GO:0005515' in x)]

In [4]:
multidomain_subs = (interpro_df[(interpro_df['type'] == 'Domain')].sort_values(by='protein_count', ascending=False))
multidomain_subs['term_hash'] = multidomain_subs['go_term'].apply(lambda x: hash(frozenset(x)) % (10 ** 8))

In [49]:
for terms in multidomain_subs['go_term']:
    print(terms)
    print(list(terms))
    break

shape: (2,)
Series: '' [str]
[
	"GO:0005524"
	"GO:0016887"
]
['GO:0005524', 'GO:0016887']


In [50]:
term_hash_map = dict(zip(multidomain_subs['term_hash'], [list(t) for t in multidomain_subs['go_term']]))

In [5]:
from Bio import SeqIO
swissprot_seq = list(SeqIO.parse('data/uniprot_sprot.fasta', 'fasta'))
swissprot_id = set([rec.id.split('|')[1] for rec in swissprot_seq])
import polars as pl
csv = pl.scan_csv("data/protein2ipr.dat", has_header=False, separator="\t")
q1 = (
    csv.rename({"column_1": "uniprot_id", "column_2": "interpro_accession"})
    .filter(pl.col("interpro_accession").is_in(set(multidomain_subs['interpro_accession'])))
    .filter(pl.col("uniprot_id").is_in(swissprot_id))
    .collect()
)
multidomain_subs = pl.from_dataframe(multidomain_subs)
multidomain_lookup = multidomain_subs[['interpro_accession', 'type', 'protein_count', 'term_hash']]

In [6]:
# Merge q1 with multidomain_lookup on interpro_accession
merged_data = q1.join(
    pl.from_dataframe(multidomain_lookup), 
    on='interpro_accession', 
    how='inner'
)

# Count unique term_hash per protein
protein_term_counts = (merged_data
    .group_by('uniprot_id')
    .agg(pl.col('term_hash').n_unique().alias('unique_term_hash_count'))
)

proteins_to_keep = protein_term_counts.filter(pl.col('unique_term_hash_count') > 1)

# Filter the merged data to keep only proteins with more than one term_hash
filtered_merged_data = merged_data.join(
    proteins_to_keep.select('uniprot_id'), 
    on='uniprot_id', 
    how='inner'
)

q = filtered_merged_data[['uniprot_id', 'interpro_accession', 'type', 'term_hash']]
q = filtered_merged_data.to_pandas()
seq_len = {rec.id.split('|')[1]: len(rec.seq) for rec in swissprot_seq}
q['seq_len'] = q['uniprot_id'].map(seq_len)
q.rename(columns={'column_5': 'domain_start', 'column_6': 'domain_end'}, inplace=True)

In [23]:
import numpy as np
from collections import defaultdict
def filter_overlapping_domains(df):
    seq_len = df['seq_len'].values
    term_hash_l = df['term_hash'].values
    domain_range_l = df[['domain_start', 'domain_end']].values
    # check all seq_len are the same
    s0 = seq_len[0]
    assert all(s == s0 for s in seq_len)
    seq_len = s0
    term_hash_dict = defaultdict(lambda: np.zeros(seq_len, dtype=bool))
    for term_hash, (si, ei) in zip(term_hash_l, domain_range_l):
        term_hash_dict[term_hash][si:ei] = True
    all_mask = np.zeros(seq_len, dtype=int)
    for term_hash in term_hash_dict:
        all_mask += term_hash_dict[term_hash]
    
    #Initialize filter_mask to series of all False
    filter_mask = pd.Series(np.zeros(len(df), dtype=bool), index=df.index)
    valid_terms = 0
    for term_hash in term_hash_dict:
        term_mask = term_hash_dict[term_hash]
        curr_mask = (all_mask - term_mask) > 0
        overlap = (curr_mask & term_mask).sum()
        if overlap > 0.2*seq_len:
            filter_mask = filter_mask & False
        curr_mask = curr_mask | term_mask
        if overlap > 0.4*term_mask.sum():
            continue
        if term_mask.sum() < 0.1*seq_len:
            continue
        valid_terms += 1
        filter_mask = filter_mask | (df['term_hash'] == term_hash)
    if valid_terms < 2:
        filter_mask = np.zeros(len(df), dtype=bool)
    return df[filter_mask]
filtered_q = q.groupby('uniprot_id').apply(filter_overlapping_domains)
filtered_q = filtered_q.reset_index(drop=True)

/tmp/ipykernel_207585/1289485648.py:37: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  filtered_q = q.groupby('uniprot_id').apply(filter_overlapping_domains)


In [27]:
def get_mask(df):
    seq_len = df['seq_len'].values[0]
    term_hash = df['term_hash'].values[0]
    domain_range_l = df[['domain_start', 'domain_end']].values
    mask = np.zeros(seq_len, dtype=bool)
    for si, ei in domain_range_l:
        mask[si:ei] = True
    return mask
r = filtered_q.groupby(['uniprot_id', 'term_hash']).apply(get_mask)
r.index = r.index.set_names(['uniprot_id', 'term_hash'])

/tmp/ipykernel_207585/2742055855.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  r = filtered_q.groupby(['uniprot_id', 'term_hash']).apply(get_mask)


In [39]:
len(set(r.index.get_level_values('term_hash')))  # number of unique term_hash
from collections import Counter
term_hash_counts = Counter(r.index.get_level_values('term_hash'))
print(term_hash_counts.most_common(10))

[(18268242, 2411), (25646312, 2286), (58577228, 2175), (26008616, 1822), (13970112, 1784), (29373422, 1713), (79966833, 1515), (15783992, 1364), (64552776, 1358), (76191735, 1337)]


In [34]:
def run_len_encode(mask):
    if not np.any(mask):
        return []
    diffs = np.diff(mask.astype(int))
    starts = np.where(diffs == 1)[0] + 1
    ends = np.where(diffs == -1)[0] + 1
    if mask[0]:
        starts = np.insert(starts, 0, 0)
    if mask[-1]:
        ends = np.append(ends, len(mask))
    return list(zip(starts, ends))

In [67]:
# Select top 10 most common term_hash
n = 15
top_n_term_hashes = [term_hash for term_hash, count in term_hash_counts.most_common(n)]
print(f"Top 25 term_hashes: {top_n_term_hashes}")
r_filtered = r[r.index.get_level_values('term_hash').isin(top_n_term_hashes)]
seq_dict = {rec.id.split('|')[1]: str(rec.seq) for rec in swissprot_seq}

Top 25 term_hashes: [18268242, 25646312, 58577228, 26008616, 13970112, 29373422, 79966833, 15783992, 64552776, 76191735, 46108113, 75151374, 52909113, 92441991, 87711439]


In [68]:
r_filtered
#Remove uniprot ids associated with only one term_hash
uid_filter = r_filtered.index.get_level_values('uniprot_id').value_counts()[lambda x: x > 1]
r_filtered = r_filtered[r_filtered.index.get_level_values('uniprot_id').isin(uid_filter.index)]

In [ ]:
term_proteins = []
for term_hash in top_n_term_hashes:
    if not term_hash in r_filtered.index.get_level_values('term_hash'):
        continue
    proteins_for_term = r_filtered.xs(term_hash, level='term_hash', drop_level=False)
    sampled_proteins = proteins_for_term.sample(n=min(50, len(proteins_for_term)), 
                                                random_state=42)
    if len(sampled_proteins) > 25:
        term_proteins.extend(sampled_proteins.index.get_level_values('uniprot_id').tolist())
term_proteins = list(set(term_proteins))

In [79]:
# Build final dataframe from term_proteins list
dataset_rows = []

for uniprot_id in term_proteins:
    # Get all term_hashes associated with this protein in r_filtered
    protein_entries = r_filtered[r_filtered.index.get_level_values('uniprot_id') == uniprot_id]
    
    for (uid, term_hash), mask in protein_entries.items():
        # Get run length encoding of mask
        annotated_indices = run_len_encode(mask)
        go_terms = term_hash_map[term_hash]
        sequence = seq_dict[uniprot_id]
        
        if sequence and annotated_indices:  # Only add if we have sequence and annotations
            dataset_rows.append({
                'UniprotID': uniprot_id,
                'AnnotatedIndices': annotated_indices,
                'GOTerm': go_terms,
                'Sequence': sequence
            })

# Create the final dataset
bidomain_dataset = pd.DataFrame(dataset_rows)
print(f"Final dataset shape: {bidomain_dataset.shape}")
print(f"Unique proteins: {bidomain_dataset['UniprotID'].nunique()}")
print(f"Total rows (protein-domain pairs): {len(bidomain_dataset)}")
print(f"\nSample of dataset:")
print(bidomain_dataset.head())

Final dataset shape: (776, 4)
Unique proteins: 388
Total rows (protein-domain pairs): 776

Sample of dataset:
  UniprotID AnnotatedIndices  \
0    Q2YAX3      [(98, 162)]   
1    Q2YAX3        [(3, 97)]   
2    Q9KXP5      [(92, 158)]   
3    Q9KXP5        [(3, 91)]   
4    A4FBC3     [(119, 557)]   

                                             GOTerm  \
0                                      [GO:0003723]   
1                                      [GO:0019843]   
2                                      [GO:0003723]   
3                                      [GO:0019843]   
4  [GO:0000166, GO:0004812, GO:0005524, GO:0006418]   

                                            Sequence  
0  MARNLDPKCRQCRREGEKLFLKGEKCFTDKCAIERRNYAPGQHGQK...  
1  MARNLDPKCRQCRREGEKLFLKGEKCFTDKCAIERRNYAPGQHGQK...  
2  MANQPRPKVKKSRALGIALTPKAVKYFEARPYPPGEHGRGRKQNSD...  
3  MANQPRPKVKKSRALGIALTPKAVKYFEARPYPPGEHGRGRKQNSD...  
4  MMRTHEAGSLRAGHAGQSVTLAGWVARRRDHGGVIFIDLRDASGVA...  


In [84]:
bidomain_dataset.to_csv('datasets/bidomain_dataset.csv', index=False, sep='\t')